# Lab Tracking trên Kaggle GPU
Chạy `scripts/run_all.py` đủ frame cho 5 video. Cần bật GPU và Internet.

**Dữ liệu:** thêm Kaggle dataset chứa `video_1` … `video_5` qua *Add Input*. Nếu không có, notebook thử tải từ Drive (hay bị chặn vì quá nhiều lượt tải).

In [ ]:
import subprocess, sys
def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

# Cài từng gói, dừng ngay khi lỗi. lapx có wheel sẵn và cung cấp module `lap`.
sh(f'{sys.executable} -m pip install -q lapx filterpy')
sh(f'{sys.executable} -m pip install -q "ultralytics>=8.4" boxmot==10.0.42')
sh('rm -rf /kaggle/working/repo && git clone -q https://github.com/quangcanh02122005/K4-Track4-Day22-Tracking-Student.git /kaggle/working/repo')

In [ ]:
import pathlib
def find_lab_data():
    for root in ('/kaggle/input', '/kaggle/working/data_lab21'):
        for p in pathlib.Path(root).rglob('video_1'):
            if p.is_dir() and (p / 'img1').is_dir():
                return p.parent
    return None

LAB_DATA = find_lab_data()
if LAB_DATA is None:  # dự phòng: Drive (có thể bị chặn)
    sh('pip install -q gdown')
    sh('cd /kaggle/working && gdown --fuzzy "https://drive.google.com/file/d/1UeVPQd6j5pSzxoJDcKJrerT9SL3vJLDt/view?usp=sharing" -O data_lab21.zip && unzip -q -o data_lab21.zip -d data_lab21 -x "__MACOSX/*"')
    LAB_DATA = find_lab_data()
assert LAB_DATA is not None, 'Không thấy video_1/img1. Hãy Add Input một Kaggle dataset chứa video_1 ... video_5.'
print('LAB_DATA =', LAB_DATA)

In [ ]:
import os
os.chdir('/kaggle/working/repo')
env = {**os.environ, 'PYTHONUTF8': '1'}
subprocess.run([sys.executable,'scripts/check_data.py','--lab-data-root',str(LAB_DATA)], check=True, env=env)

In [ ]:
# Bản nộp: đủ frame, GPU. Sửa configs/nop_bai.json trong repo rồi push để đổi tracker / conf / iou.
subprocess.run([sys.executable,'scripts/run_all.py','--lab-data-root',str(LAB_DATA),
                '--config','configs/nop_bai.json','--out','/kaggle/working/runs/nop_bai',
                '--device','cuda:0','--save-video'], check=True, env=env)

In [ ]:
sh('ls -la /kaggle/working/runs/nop_bai')